# **Chemprop Model Training and Evaluation**

This notebook trains and evaluates Chemprop regression models for predicting pIC50 values for each target using two input strategies: SMILES-only and SMILES with additional RFECV-selected descriptor features. The workflow loads the optimized hyperparameters, trains five replicate models for robustness, generates predictions on the external test set, and computes performance metrics including R², RMSE, and MAE. Results are saved for both individual replicates and ensemble predictions to support comparison of Chemprop model performance across targets.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
# ============================================================
# 1. Install and import Chemprop
# ============================================================

import os
import glob
import shutil
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

if os.getenv("COLAB_RELEASE_TAG"):
    try:
        import chemprop  # noqa: F401
    except ImportError:
        !git clone https://github.com/chemprop/chemprop.git
        %cd chemprop
        !pip install .
        !pip install -U ray[tune]
        !pip install -e .[hpopt]
        %cd /content

In [ ]:
# ============================================================
# 2. Define target and paths
# ============================================================

targets = ["5-HT6", "ache", "bace1", "buche", "esr1", "gsk-3beta", "mao-b"]
target = targets[2]   # change target here

base = "/content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/chemprop"

DATA_PATH = f"{base}/input_data/{target}_train.csv"
TEST_PATH = f"{base}/input_data/{target}_test.csv"

DESC_TRAIN_CSV = f"{base}/input_data/reu_features/{target}_train_reu.csv"
DESC_TEST_CSV = f"{base}/input_data/reu_features/{target}_test_reu.csv"

DESC_TRAIN_NPZ = DESC_TRAIN_CSV.replace(".csv", ".npz")
DESC_TEST_NPZ = DESC_TEST_CSV.replace(".csv", ".npz")

CONFIG_TOML = f"{base}/hyperopt_models/{target}/best_config.toml"

SMILES_LOCAL_OUT = f"/content/chemprop_runs/{target}_smiles_only"
FEATURE_LOCAL_OUT = f"/content/chemprop_runs/{target}_features"

SMILES_DRIVE_OUT = f"{base}/training/smiles_only_model/{target}"
FEATURE_DRIVE_OUT = f"{base}/training/feature_model/{target}"

LABEL_COL = "pIC50"

In [ ]:
# ============================================================
# 3. Sanity checks
# ============================================================

required_files = [
    DATA_PATH,
    TEST_PATH,
    DESC_TRAIN_CSV,
    DESC_TEST_CSV,
    CONFIG_TOML,
]

for file_path in required_files:
    assert os.path.exists(file_path), f"Missing file: {file_path}"

assert LABEL_COL in pd.read_csv(DATA_PATH, nrows=1).columns, f"{LABEL_COL} missing in train file"
assert LABEL_COL in pd.read_csv(TEST_PATH, nrows=1).columns, f"{LABEL_COL} missing in test file"

print("All required files found.")

In [ ]:

# ============================================================
# 4. Convert descriptor CSV files to NPZ format
# ============================================================

Xtr = pd.read_csv(DESC_TRAIN_CSV).to_numpy(dtype=np.float32)
Xte = pd.read_csv(DESC_TEST_CSV).to_numpy(dtype=np.float32)

np.savez_compressed(DESC_TRAIN_NPZ, Xtr)
np.savez_compressed(DESC_TEST_NPZ, Xte)

n_train = sum(1 for _ in open(DATA_PATH, "r", encoding="utf-8")) - 1
n_test = sum(1 for _ in open(TEST_PATH, "r", encoding="utf-8")) - 1

assert n_train == Xtr.shape[0], "Training row count mismatch between CSV and descriptor file"
assert n_test == Xte.shape[0], "Test row count mismatch between CSV and descriptor file"

print("Descriptor NPZ files saved:")
print(" -", DESC_TRAIN_NPZ, Xtr.shape)
print(" -", DESC_TEST_NPZ, Xte.shape)
print("Row counts verified.")


In [ ]:


# ============================================================
# 5. Helper functions
# ============================================================

def run_command(cmd):
    """Run a command and raise an error if it fails."""
    print("Running:", " ".join(cmd))
    proc = subprocess.run(cmd, capture_output=True, text=True)
    print(proc.stdout)
    print(proc.stderr)
    if proc.returncode != 0:
        raise RuntimeError(f"Command failed: {' '.join(cmd)}")


def copy_to_drive(local_dir, drive_dir):
    """Copy training artifacts from local runtime to Google Drive."""
    os.makedirs(drive_dir, exist_ok=True)
    for item in os.listdir(local_dir):
        src = os.path.join(local_dir, item)
        dst = os.path.join(drive_dir, item)
        if os.path.isdir(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
        else:
            shutil.copy2(src, dst)
    print(f"Copied artifacts to: {drive_dir}")


def get_checkpoints(model_dir):
    """Collect one checkpoint per replicate, preferring best*.ckpt over last.ckpt."""
    ckpts = []
    replicate_dirs = sorted(
        glob.glob(os.path.join(model_dir, "replicate_*")),
        key=lambda path: int(Path(path).name.split("_")[-1])
    )

    for rep in replicate_dirs:
        bests = sorted(
            glob.glob(os.path.join(rep, "**", "checkpoints", "best*.ckpt"), recursive=True)
        )
        lasts = sorted(
            glob.glob(os.path.join(rep, "**", "checkpoints", "last.ckpt"), recursive=True)
        )

        if bests:
            ckpts.append(bests[-1])
        elif lasts:
            ckpts.append(lasts[-1])

    if not ckpts:
        raise FileNotFoundError(f"No checkpoints found under {model_dir}")

    return ckpts


def extract_prediction_column(df):
    """Return the prediction column from a Chemprop prediction dataframe."""
    if "prediction" in df.columns:
        return df["prediction"].values

    pred_cols = [col for col in df.columns if "pred" in col.lower()]
    if not pred_cols:
        raise ValueError(f"No prediction column found. Columns: {df.columns.tolist()}")

    return df[pred_cols[0]].values


def evaluate_predictions(test_path, pred_files, ensemble_csv, target_name, summary_csv):
    """Compute ensemble and replicate metrics and save them."""
    y_true = pd.read_csv(test_path)[LABEL_COL].values

    # Ensemble metrics
    df_ens = pd.read_csv(ensemble_csv)
    y_pred_ens = extract_prediction_column(df_ens)

    ens_r2 = float(r2_score(y_true, y_pred_ens))
    ens_mse = float(mean_squared_error(y_true, y_pred_ens))
    ens_rmse = float(np.sqrt(ens_mse))
    ens_mae = float(mean_absolute_error(y_true, y_pred_ens))

    # Replicate metrics
    r2s, rmses, maes = [], [], []

    for pred_file in pred_files:
        df_pred = pd.read_csv(pred_file)
        y_pred = extract_prediction_column(df_pred)

        r2s.append(r2_score(y_true, y_pred))
        mse = mean_squared_error(y_true, y_pred)
        rmses.append(np.sqrt(mse))
        maes.append(mean_absolute_error(y_true, y_pred))

    avg_r2 = float(np.mean(r2s))
    std_r2 = float(np.std(r2s, ddof=0))
    avg_rmse = float(np.mean(rmses))
    std_rmse = float(np.std(rmses, ddof=0))
    avg_mae = float(np.mean(maes))
    std_mae = float(np.std(maes, ddof=0))

    print(f"\nEnsemble  → R²={ens_r2:.4f}  RMSE={ens_rmse:.4f}  MAE={ens_mae:.4f}")
    print(
        f"Replicate → R²={avg_r2:.4f}±{std_r2:.4f}  "
        f"RMSE={avg_rmse:.4f}±{std_rmse:.4f}  "
        f"MAE={avg_mae:.4f}±{std_mae:.4f}"
    )

    row = pd.DataFrame([{
        "Target": target_name,
        "Replicates": len(pred_files),
        "ENS_R2": ens_r2,
        "ENS_RMSE": ens_rmse,
        "ENS_MAE": ens_mae,
        "Avg_R2": avg_r2,
        "Std_R2": std_r2,
        "Avg_RMSE": avg_rmse,
        "Std_RMSE": std_rmse,
        "Avg_MAE": avg_mae,
        "Std_MAE": std_mae,
    }])

    if os.path.exists(summary_csv):
        existing = pd.read_csv(summary_csv)
        row = pd.concat([existing, row], ignore_index=True)

    row.to_csv(summary_csv, index=False)
    print("Saved summary to:", summary_csv)


def predict_with_checkpoints(model_dir, test_path, output_name, descriptors_path=None):
    """Generate per-replicate and ensemble predictions."""
    ckpts = get_checkpoints(model_dir)

    print("Using checkpoints:")
    for ckpt in ckpts:
        print(" -", ckpt)

    pred_files = []

    for ckpt in ckpts:
        out_csv = ckpt + ".preds.csv"
        if os.path.exists(out_csv):
            os.remove(out_csv)

        cmd = [
            "chemprop", "predict",
            "--test-path", test_path,
            "--smiles-columns", "smiles",
            "--model-paths", ckpt,
            "--output", out_csv,
        ]

        if descriptors_path is not None:
            cmd.extend(["--descriptors-path", descriptors_path])

        run_command(cmd)
        pred_files.append(out_csv)

    ensemble_csv = os.path.join(model_dir, output_name)
    if os.path.exists(ensemble_csv):
        os.remove(ensemble_csv)

    ensemble_cmd = [
        "chemprop", "predict",
        "--test-path", test_path,
        "--smiles-columns", "smiles",
        "--model-paths", *ckpts,
        "--output", ensemble_csv,
    ]

    if descriptors_path is not None:
        ensemble_cmd.extend(["--descriptors-path", descriptors_path])

    run_command(ensemble_cmd)

    return pred_files, ensemble_csv


In [ ]:
# ============================================================
# 6. Train SMILES-only Chemprop model
# ============================================================

os.makedirs(SMILES_LOCAL_OUT, exist_ok=True)

smiles_train_cmd = [
    "chemprop", "train",
    "--data-path", DATA_PATH,
    "--task-type", "regression",
    "--config-path", CONFIG_TOML,
    "--num-replicates", "5",
    "--data-seed", "0",
    "--save-smiles-splits",
    "--accelerator", "gpu",
    "--devices", "1",
    "--output-dir", SMILES_LOCAL_OUT,
    "--logfile",
]

run_command(smiles_train_cmd)
copy_to_drive(SMILES_LOCAL_OUT, SMILES_DRIVE_OUT)

In [ ]:
# ============================================================
# 7. Train Chemprop model with additional descriptor features
# ============================================================

os.makedirs(FEATURE_LOCAL_OUT, exist_ok=True)

feature_train_cmd = [
    "chemprop", "train",
    "--data-path", DATA_PATH,
    "--task-type", "regression",
    "--config-path", CONFIG_TOML,
    "--descriptors-path", DESC_TRAIN_NPZ,
    "--num-replicates", "5",
    "--data-seed", "0",
    "--save-smiles-splits",
    "--accelerator", "gpu",
    "--devices", "1",
    "--output-dir", FEATURE_LOCAL_OUT,
    "--logfile",
]

run_command(feature_train_cmd)
copy_to_drive(FEATURE_LOCAL_OUT, FEATURE_DRIVE_OUT)

In [ ]:
# ============================================================
# 8. Evaluate SMILES-only model
# ============================================================

smiles_pred_files, smiles_ensemble_csv = predict_with_checkpoints(
    model_dir=SMILES_DRIVE_OUT,
    test_path=TEST_PATH,
    output_name="ensemble_preds_5rep.csv",
    descriptors_path=None,
)

smiles_summary_csv = os.path.join(SMILES_DRIVE_OUT, "replicate_metrics_summary.csv")

evaluate_predictions(
    test_path=TEST_PATH,
    pred_files=smiles_pred_files,
    ensemble_csv=smiles_ensemble_csv,
    target_name=target,
    summary_csv=smiles_summary_csv,
)

In [ ]:
# ============================================================
# 9. Evaluate feature-based model
# ============================================================

feature_pred_files, feature_ensemble_csv = predict_with_checkpoints(
    model_dir=FEATURE_DRIVE_OUT,
    test_path=TEST_PATH,
    output_name="ensemble_preds_5rep.csv",
    descriptors_path=DESC_TEST_NPZ,
)

feature_summary_csv = os.path.join(FEATURE_DRIVE_OUT, "replicate_metrics_summary.csv")

evaluate_predictions(
    test_path=TEST_PATH,
    pred_files=feature_pred_files,
    ensemble_csv=feature_ensemble_csv,
    target_name=target,
    summary_csv=feature_summary_csv,
)